#Segmentation of urban scenes

##CP1. Data processing and preparation / Обработка и подготовка данных

###Importing all modules (no need to use in CP1)

In [ ]:
import torch
import timm
from PIL import Image
from urllib.request import urlopen
import multiprocessing
from torchmetrics import JaccardIndex

from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.nn.functional as F
import torchvision
from transformers import AutoModel
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

from zipfile import ZipFile
import os
from google.colab import userdata


username = "miloskutov@edu.hse.ru"
password = userdata.get('CSDatasetPassword')

###Getting datset and unpcking zip-files / Получение данных и распаковка zip-файлов

In [29]:
import requests
import os
import zipfile
from getpass import getpass
from typing import Optional, List, Tuple

import torch
from torch.utils.data import Dataset, DataLoader

from google.colab import userdata

USERNAME = "miloskutov@edu.hse.ru"
PASSWORD = userdata.get('CSDatasetPassword')

CITYSCAPES_PACKAGES = {
    'gtFine': {
        'package_id': '1',
        'filename': 'gtFine_trainvaltest.zip',
        'description': 'Аннотации (маски) высокого качества для train/val/test'
    },
    'leftImg8bit': {
        'package_id': '3',
        'filename': 'leftImg8bit_trainvaltest.zip',
        'description': 'Исходные изображения (RGB) для train/val/test'
    },
}

BASE_URL = "https://www.cityscapes-dataset.com"
LOGIN_URL = f"{BASE_URL}/login/"
DOWNLOAD_URL_TEMPLATE = f"{BASE_URL}/file-handling/?packageID={{}}"

CITYSCAPES_COLORS = {
    0: (128, 64, 128),      # road
    1: (244, 35, 232),      # sidewalk
    2: (70, 70, 70),        # building
    3: (102, 102, 156),     # wall
    4: (190, 153, 153),     # fence
    5: (153, 153, 153),     # pole
    6: (250, 170, 30),      # traffic light
    7: (220, 220, 0),       # traffic sign
    8: (107, 142, 35),      # vegetation
    9: (152, 251, 152),     # terrain
    10: (70, 130, 180),     # sky
    11: (220, 20, 60),      # person
    12: (255, 0, 0),        # rider
    13: (0, 0, 142),        # car
    14: (0, 0, 70),         # truck
    15: (0, 60, 100),       # bus
    16: (0, 80, 100),       # train
    17: (0, 0, 230),        # motorcycle
    18: (119, 11, 32),      # bicycle
}

CITYSCAPES_CLASS_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain', 'sky',
    'person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle'
]

In [3]:
def authenticate_cityscapes(username: str, password: str) -> requests.Session:
    session = requests.Session()

    response = session.get(LOGIN_URL)

    login_data = {
        'username': username,
        'password': password,
        'submit': 'Login'
    }

    login_response = session.post(LOGIN_URL, data=login_data)

    if "Logout" not in login_response.text:
        raise ValueError(
            "Ошибка авторизации! Проверьте логин и пароль. "
            "Убедитесь, что аккаунт подтверждён на cityscapes-dataset.com"
        )

    return session

In [4]:
def download_package(
    session: requests.Session,
    package_key: str,
    download_dir: str,
    chunk_size: int = 8192
) -> str:
    if package_key not in CITYSCAPES_PACKAGES:
        raise ValueError(f"Неизвестный пакет: {package_key}. Доступные: {list(CITYSCAPES_PACKAGES.keys())}")

    package_info = CITYSCAPES_PACKAGES[package_key]
    download_url = DOWNLOAD_URL_TEMPLATE.format(package_info['package_id'])
    output_path = os.path.join(download_dir, package_info['filename'])

    if os.path.exists(output_path):
        print(f"Файл уже существует: {output_path}. Пропускаем загрузку.")
        return output_path

    print(f"Загрузка: {package_info['description']}")
    print(f"URL: {download_url}")
    print(f"Сохранение в: {output_path}")

    with session.get(download_url, stream=True) as response:
        response.raise_for_status()

        total_size = int(response.headers.get('content-length', 0))
        downloaded = 0

        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size:
                        percent = downloaded / total_size * 100
                        print(f"\rПрогресс: {percent:.1f}%", end='', flush=True)

        print(f"\nЗагрузка завершена: {output_path}")

    return output_path

In [5]:
def extract_zip(zip_path: str, extract_to: str, delete_after: bool = False) -> List[str]:
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Архив не найден: {zip_path}")

    print(f"Распаковка: {zip_path}")
    print(f"Целевая директория: {extract_to}")

    extracted_items = []

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        print(f"Файлов в архиве: {len(file_list)}")

        zip_ref.extractall(extract_to)

        extracted_items = list(set(os.path.dirname(f) or f for f in file_list))

    print(f"Распаковка завершена. Извлечено элементов: {len(extracted_items)}")

    if delete_after:
        os.remove(zip_path)
        print(f"Архив удалён: {zip_path}")

    return extracted_items

In [6]:
def download_cityscapes_dataset(
    packages: List[str],
    download_dir: str,
    username: Optional[str] = None,
    password: Optional[str] = None,
    extract: bool = True,
    delete_zips: bool = False
) -> dict:
    os.makedirs(download_dir, exist_ok=True)

    if username is None:
        username = input("Введите логин CityScapes: ").strip()
    if password is None:
        password = getpass("Введите пароль (не отображается): ")

    session = authenticate_cityscapes(username, password)
    print("Авторизация успешна")

    results = {}

    for package_key in packages:
        print(f"\n--- Обработка пакета: {package_key} ---")

        zip_path = download_package(session, package_key, download_dir)
        results[package_key] = {'zip_path': zip_path}

        if extract:
            extracted = extract_zip(zip_path, download_dir, delete_after=delete_zips)
            results[package_key]['extracted_items'] = extracted

    return results

In [7]:
def verify_cityscapes_structure(base_dir: str) -> Tuple[bool, dict]:
    report = {'errors': [], 'warnings': [], 'found_dirs': []}

    expected_dirs = ['gtFine', 'leftImg8bit']
    for dirname in expected_dirs:
        dir_path = os.path.join(base_dir, dirname)
        if os.path.isdir(dir_path):
            report['found_dirs'].append(dirname)

            for split in ['train', 'val']:
                split_path = os.path.join(dir_path, split)
                if not os.path.isdir(split_path):
                    report['warnings'].append(
                        f"Отсутствует ожидаемая поддиректория: {split_path}"
                    )
        else:
            report['errors'].append(f"Отсутствует обязательная директория: {dir_path}")

    is_valid = len(report['errors']) == 0

    if is_valid:
        print("Структура датасета: ОК")
    else:
        print("Ошибки в структуре датасета:")
        for err in report['errors']:
            print(f"  - {err}")

    return is_valid, report

In [ ]:
!mkdir -p /content/cityscapes

In [ ]:
import requests
import os
from getpass import getpass

# 2. Создаём сессию
session = requests.Session()

# 3. Получаем CSRF-токен (если требуется)
login_url = "https://www.cityscapes-dataset.com/login/"
response = session.get(login_url)

# Иногда требуется парсить токен, но часто достаточно просто POST
login_data = {
    'username': username,
    'password': password,
    'submit': 'Login'
}

# 4. Авторизуемся
login_response = session.post(login_url, data=login_data)

if "Logout" not in login_response.text:
    raise ValueError("Ошибка входа! Проверьте email и пароль.")

print("✅ Успешная авторизация")

# 5. Скачиваем файл (например, gtFine)
os.makedirs('/content/cityscapes', exist_ok=True)

package_url = "https://www.cityscapes-dataset.com/file-handling/?packageID=1"
output_path = "/content/cityscapes/gtFine_trainvaltest.zip"

print("📥 Начинаем загрузку gtFine...")

with session.get(package_url, stream=True) as r:
    r.raise_for_status()
    with open(output_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("✅ Загрузка завершена")

In [ ]:
package_url = "https://www.cityscapes-dataset.com/file-handling/?packageID=3"
output_path = "/content/cityscapes/leftImg8bit_trainvaltest.zip"

print("📥 Скачиваем исходные изображения...")

with session.get(package_url, stream=True) as r:
    r.raise_for_status()
    with open(output_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("✅ Изображения загружены")

In [ ]:
with ZipFile('/content/cityscapes/gtFine_trainvaltest.zip', 'r') as file:
  file.extractall('/content/cityscapes')

with ZipFile('/content/cityscapes/leftImg8bit_trainvaltest.zip', 'r') as file:
  file.extractall('/content/cityscapes')

###Data analysis / Аналитика данных

Функция для проверки соответствия изображений и масок, а именно их имён, распределения классов и визуализация распределения

In [20]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def check_dataset_integrity(img_dir: str, mask_dir: str) -> dict:
    print(f"\n--- Проверка целостности датасета ---")
    print(f"Изображения: {img_dir}")
    print(f"Маски: {mask_dir}")

    results = {
        'total_images': 0,
        'total_masks': 0,
        'matching_pairs': 0,
        'missing_masks': [],
        'missing_images': [],
        'class_distribution': np.zeros(35),
        'errors': []
    }

    try:
        img_files = set(os.listdir(img_dir))
        mask_files = set(os.listdir(mask_dir))
    except FileNotFoundError as e:
        results['errors'].append(f"Директория не найдена: {e}")
        return results

    img_names = {f.replace('_leftImg8bit.png', '') for f in img_files if f.endswith('_leftImg8bit.png')}
    mask_names = {f.replace('_gtFine_labelIds.png', '') for f in mask_files if f.endswith('_gtFine_labelIds.png')}

    results['total_images'] = len(img_names)
    results['total_masks'] = len(mask_names)

    missing_masks = img_names - mask_names
    missing_images = mask_names - img_names

    results['missing_masks'] = list(missing_masks)[:10]
    results['missing_images'] = list(missing_images)[:10]
    results['matching_pairs'] = len(img_names & mask_names)

    print(f"Всего изображений: {results['total_images']}")
    print(f"Всего масок: {results['total_masks']}")
    print(f"Совпадающих пар: {results['matching_pairs']}")

    if missing_masks:
        print(f"Предупреждение: Отсутствуют маски для {len(missing_masks)} изображений")
    if missing_images:
        print(f"Предупреждение: Отсутствуют изображения для {len(missing_images)} масок")

    print("\nАнализ распределения классов...")
    class_counts = np.zeros(35)
    total_pixels = 0

    for mask_file in mask_files:
        if not mask_file.endswith('_gtFine_labelIds.png'):
            continue

        mask_path = os.path.join(mask_dir, mask_file)
        try:
            mask = Image.open(mask_path)
            mask_arr = np.array(mask)
            unique, counts = np.unique(mask_arr, return_counts=True)

            for class_id, count in zip(unique, counts):
                if class_id < 35:
                    class_counts[class_id] += count
                    total_pixels += count
        except Exception as e:
            results['errors'].append(f"Ошибка при чтении маски {mask_file}: {e}")

    results['class_distribution'] = class_counts

    print("\nРаспределение классов (первые 19):")
    print(f"{'Класс':<20} {'Пикселей':>12} {'Процент':>10}")
    print("-" * 45)

    for class_id in range(19):
        count = int(class_counts[class_id])
        percentage = (count / total_pixels * 100) if total_pixels > 0 else 0
        class_name = CITYSCAPES_CLASS_NAMES[class_id]
        print(f"{class_name:<20} {count:>12,} {percentage:>9.2f}%")

    if total_pixels > 0:
        max_class = np.argmax(class_counts[:19])
        min_class = np.argmin([c for c in class_counts[:19] if c > 0])
        ratio = class_counts[max_class] / class_counts[min_class] if class_counts[min_class] > 0 else float('inf')
        print(f"\nДисбаланс классов: самый частый '{CITYSCAPES_CLASS_NAMES[max_class]}' "
              f"в {ratio:.1f} раз чаще самого редкого")

    return results

In [21]:
def visualize_sample(img_path: str, mask_path: str, output_path: Optional[str] = None):
    print(f"\n--- Визуализация образца ---")
    print(f"Изображение: {img_path}")
    print(f"Маска: {mask_path}")

    img = Image.open(img_path).convert('RGB')
    mask = Image.open(mask_path)
    mask_arr = np.array(mask)

    color_mask = np.zeros((mask_arr.shape[0], mask_arr.shape[1], 3), dtype=np.uint8)
    for class_id, color in CITYSCAPES_COLORS.items():
        if class_id < 19:
            color_mask[mask_arr == class_id] = color

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(img)
    axes[0].set_title("Исходное изображение", fontsize=10, fontweight='bold')
    axes[0].axis('off')

    im1 = axes[1].imshow(mask_arr, cmap='gray', vmin=0, vmax=18)
    axes[1].set_title("Маска (grayscale)", fontsize=10, fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(color_mask)
    axes[2].set_title("Цветная сегментация", fontsize=10, fontweight='bold')
    axes[2].axis('off')

    overlay = img.copy()
    overlay_array = np.array(overlay)
    alpha = 0.5
    blend = (overlay_array * (1 - alpha) + color_mask * alpha).astype(np.uint8)
    axes[3].imshow(blend)
    axes[3].set_title("Наложение (alpha=0.5)", fontsize=10, fontweight='bold')
    axes[3].axis('off')

    plt.suptitle(f"CityScapes Sample: {os.path.basename(img_path)}", fontsize=12, fontweight='bold')
    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Сохранено: {output_path}")

    plt.show()

    unique, counts = np.unique(mask_arr, return_counts=True)
    print("\nКлассы в данной маске:")
    total_pixels = mask_arr.size
    for class_id, count in zip(unique, counts):
        if class_id < 19 and count > 100:
            percentage = count / total_pixels * 100
            class_name = CITYSCAPES_CLASS_NAMES[class_id]
            print(f"  {class_name:<15}: {count:>8,} пикселей ({percentage:5.2f}%)")

In [22]:
def find_sample_pair(base_dir: str, split: str = 'train') -> Tuple[str, str]:
    img_split_dir = os.path.join(base_dir, 'leftImg8bit', split)
    mask_split_dir = os.path.join(base_dir, 'gtFine', split)

    if not os.path.exists(img_split_dir) or not os.path.exists(mask_split_dir):
        raise FileNotFoundError(f"Директории сплита '{split}' не найдены")

    cities = os.listdir(img_split_dir)
    for city in cities:
        city_img_dir = os.path.join(img_split_dir, city)
        city_mask_dir = os.path.join(mask_split_dir, city)

        if os.path.isdir(city_img_dir) and os.path.isdir(city_mask_dir):
            img_files = [f for f in os.listdir(city_img_dir) if f.endswith('_leftImg8bit.png')]
            if img_files:
                img_name = img_files[0]
                base_name = img_name.replace('_leftImg8bit.png', '')

                img_path = os.path.join(city_img_dir, img_name)
                mask_path = os.path.join(city_mask_dir, f"{base_name}_gtFine_labelIds.png")

                if os.path.exists(mask_path):
                    return img_path, mask_path

    raise FileNotFoundError(f"Не найдено пар изображение-маска в сплите '{split}'")

In [23]:
def plot_class_distribution_bar(mask_dir: str, output_path: Optional[str] = None):
    print(f"\n--- Анализ распределения классов ---")
    print(f"Директория с масками: {mask_dir}")

    class_pixels = np.zeros(19)
    total_masks = 0

    for root, dirs, files in os.walk(mask_dir):
        for file in files:
            if file.endswith('_gtFine_labelIds.png'):
                mask_path = os.path.join(root, file)
                try:
                    mask = np.array(Image.open(mask_path))
                    for class_id in range(19):
                        class_pixels[class_id] += np.sum(mask == class_id)
                    total_masks += 1
                except Exception as e:
                    print(f"Ошибка при чтении {file}: {e}")

    print(f"Обработано масок: {total_masks}")

    total_pixels = class_pixels.sum()
    percentages = (class_pixels / total_pixels * 100) if total_pixels > 0 else np.zeros(19)

    plt.figure(figsize=(14, 7))
    bars = plt.bar(range(len(CITYSCAPES_CLASS_NAMES)), percentages, color='steelblue', alpha=0.7)

    plt.xticks(range(len(CITYSCAPES_CLASS_NAMES)), CITYSCAPES_CLASS_NAMES, rotation=45, ha='right', fontsize=8)
    plt.ylabel("Процент пикселей (%)", fontsize=10)
    plt.title("Распределение классов в CityScapes", fontsize=12, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)

    for bar, pct in zip(bars, percentages):
        if pct > 0.1:
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{pct:.1f}%', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"График сохранён: {output_path}")

    plt.show()

    sorted_idx = np.argsort(percentages)[::-1]
    print("\nТоп-5 самых частых классов:")
    for i in sorted_idx[:5]:
        print(f"  {i+1}. {CITYSCAPES_CLASS_NAMES[i]}: {percentages[i]:.2f}%")

    print("\nТоп-5 самых редких классов:")
    for i in sorted_idx[-5:][::-1]:
        print(f"  {i+1}. {CITYSCAPES_CLASS_NAMES[i]}: {percentages[i]:.2f}%")

###Transform and dataloaders / Аугментация данных при помощи transform (albumentations.Compose()) и создание даталоадеров для обучения моделей

In [10]:
def transform_A(data_config):
  input_size = data_config['input_size']
  mean = data_config['mean']
  std = data_config['std']
  scale = (0.008, 1.0) #data_config['scale']
  ratio = (0.75, 1.3333) #data_config['ratio']

  crop_pct = data_config.get('crop_pct', 1.0)
  resize_size1 = int(input_size[1] / crop_pct)
  resize_size2 = int(input_size[2] / crop_pct)

  train_transform = A.Compose([
      A.RandomResizedCrop(
          size=(resize_size1, resize_size2),
          scale=scale,
          ratio=ratio,
          interpolation=cv2.INTER_CUBIC,
          p=1.0
      ),
      A.HorizontalFlip(p=0.5),
      A.ColorJitter(
          brightness=(0.6, 1.4),
          contrast=(0.6, 1.4),
          saturation=(0.6, 1.4),
          hue=0.0,
          p=1.0
      ),
      A.Normalize(mean=mean, std=std, max_pixel_value=255.0),
      ToTensorV2()
  ], additional_targets={'mask': 'mask'})

  val_transform = A.Compose([
      A.Resize(height=resize_size1, width=resize_size2, interpolation=cv2.INTER_CUBIC),
      A.Normalize(mean=mean, std=std),
      ToTensorV2()
  ], additional_targets={'mask': 'mask'})


  return train_transform, val_transform

In [11]:
def create_data_loaders(
    root,
    train_transform=None,
    val_transform=None,
    id_to_trainid=None,
    batch_size=1,
    is_test=True
):
  train_dataset = CSDataset(
      root,
      split='train',
      transform=train_transform,
      id_to_trainid=id_to_trainid
  )
  train_loader = DataLoader(
      train_dataset,
      batch_size=batch_size,
      shuffle=True,
      num_workers=2,
      pin_memory=True,
      drop_last=True
  )

  val_dataset = CSDataset(
      root,
      split='val',
      transform=val_transform,
      id_to_trainid=id_to_trainid
  )
  val_loader = DataLoader(
      val_dataset,
      batch_size=batch_size,
      shuffle=False,
      num_workers=2,
      pin_memory=True
  )

  test_dataset = CSDataset(
      root,
      split='test',
      transform=val_transform,
      id_to_trainid=id_to_trainid
  )
  test_loader = DataLoader(
      test_dataset,
      batch_size=batch_size,
      shuffle=False,
      num_workers=2,
      pin_memory=True
  )

  if is_test:
    test_loaders(train_loader, val_loader, test_loader)


  return train_loader, val_loader, test_loader

Testing correct work of all dataloaders/ Функция для проверки корректной работы всех созданных dataloader-ов

In [12]:
def test_loaders(train_loader, val_loader, test_loader):

  for img, tgt in tqdm(train_loader):
    # print(img)
    print('Train image:', img.shape, img.dtype)
    print('Train target:', tgt.shape, tgt.dtype)
    print('Classes:', torch.unique(tgt))
    break

  for img, tgt in tqdm(val_loader):
    print('Val image:', img.shape, img.dtype)
    print('Val target:', tgt.shape, tgt.dtype)
    print('Classes:', torch.unique(tgt))
    break

  for img, tgt in tqdm(test_loader):
    print('Test image:', img.shape, img.dtype)
    print('Test target:', tgt.shape, tgt.dtype)
    print('Classes:', torch.unique(tgt))
    break

In [13]:
def test_data_loader(
    image_path,
    target_path,
    val_transform=None,
    num_imgs=2,
    batch_size=1,
    id_to_trainid=None
):
  single_image = SingleImageDataset(
      image_path, target_path, num_imgs=num_imgs,
      transform=val_transform, id_to_trainid=id_to_trainid
  )

  image_loader = DataLoader(
      single_image, batch_size=batch_size, shuffle=False, pin_memory=True
  )

  return image_loader

###Custom datasets / Переопределённые классы Dataset для кастомизации работы с изображениями из CityScapes датасета

In [17]:
class CSDataset(Dataset):
  def __init__(self, root, split='train', transform=None, id_to_trainid=None):
    self.root = root
    self.split = split
    self.transform = transform
    self.id_to_trainid = id_to_trainid

    self.images = []
    self.targets = []

    img_dir = os.path.join(root, 'leftImg8bit', split)
    target_dir = os.path.join(root, 'gtFine', split)

    for city in sorted(os.listdir(img_dir)):
      img_city_dir = os.path.join(img_dir, city)
      target_city_dir = os.path.join(target_dir, city)

      for fname in sorted(os.listdir(img_city_dir)):
        if fname.endswith('_leftImg8bit.png'):
          prefix = fname.replace('_leftImg8bit.png', '')
          self.images.append(os.path.join(img_city_dir, fname))
          self.targets.append(os.path.join(
              target_city_dir,
              prefix + '_gtFine_labelIds.png'
            ))
    print(self.images[0], self.targets[0])


  def __len__(self):
    return len(self.images)


  def __getitem__(self, idx):
    image = np.array(Image.open(self.images[idx]).convert('RGB'))
    target = np.array(Image.open(self.targets[idx]))
    # print(image, target)

    if self.transform:
      transformed = self.transform(image=image, mask=target)
      image = transformed['image']
      target = transformed['mask']

    # print(image, target)

    if self.id_to_trainid is not None:
      target = self.id_to_trainid[target]
      target = torch.from_numpy(target).long()
    # print(image, target)


    return image, target

Dataset for testing training of models / Датасет для тестирования обучения моделей

In [18]:
class SingleImageDataset(Dataset):
  def __init__(self, image_path, target_path, num_imgs=2, transform=None, id_to_trainid=None):
    self.image_path = image_path
    self.target_path = target_path
    self.transform = transform
    self.id_to_trainid = id_to_trainid
    self.num_imgs=num_imgs

  def __len__(self):
    return self.num_imgs

  def __getitem__(self, idx):
    image = np.array(Image.open(self.image_path).convert('RGB'))
    target = np.array(Image.open(self.target_path))

    if self.transform:
      transformed = self.transform(image=image, mask=target)
      image = transformed['image']
      target = transformed['mask']

    if self.id_to_trainid is not None:
      target_mapped = np.copy(target)
      for k, v in self.id_to_trainid.items():
        target_mapped[target == k] = v
      target = torch.from_numpy(target_mapped).long()
    else:
        target = torch.from_numpy(target).long()

    return image, target

In [ ]:
if __name__ == "__main__":
    DOWNLOAD_DIR = "/content/cityscapes"
    PACKAGES_TO_DOWNLOAD = ['gtFine', 'leftImg8bit']

    print("=" * 60)
    print("CityScapes Dataset Downloader & Analyzer")
    print("=" * 60)

    print("\n[ШАГ 1] Скачивание и распаковка датасета")
    print("-" * 60)

    results = download_cityscapes_dataset(
        packages=PACKAGES_TO_DOWNLOAD,
        download_dir=DOWNLOAD_DIR,
        username=USERNAME,
        password=PASSWORD,
        extract=True,
        delete_zips=False
    )

    print("\n[ШАГ 2] Проверка структуры датасета")
    print("-" * 60)

    is_valid, report = verify_cityscapes_structure(DOWNLOAD_DIR)

    if report['warnings']:
        print("\nПредупреждения:")
        for w in report['warnings']:
            print(f"  ! {w}")

    if not is_valid:
        print("\nКритические ошибки! Остановка.")
        exit(1)

    print("\n[ШАГ 3] Проверка целостности данных")
    print("-" * 60)

    try:
        train_img_dir = os.path.join(DOWNLOAD_DIR, 'leftImg8bit', 'train')
        train_mask_dir = os.path.join(DOWNLOAD_DIR, 'gtFine', 'train')

        cities = os.listdir(train_img_dir)
        if cities:
            first_city = cities[0]
            city_img_path = os.path.join(train_img_dir, first_city)
            city_mask_path = os.path.join(train_mask_dir, first_city)

            integrity_results = check_dataset_integrity(city_img_path, city_mask_path)
    except Exception as e:
        print(f"Ошибка при проверке целостности: {e}")

    print("\n[ШАГ 4] Анализ распределения классов")
    print("-" * 60)

    try:
        mask_train_dir = os.path.join(DOWNLOAD_DIR, 'gtFine', 'train')
        plot_class_distribution_bar(
            mask_train_dir,
            output_path=os.path.join(DOWNLOAD_DIR, 'class_distribution.png')
        )
    except Exception as e:
        print(f"Ошибка при анализе распределения: {e}")

    print("\n[ШАГ 5] Визуализация примеров")
    print("-" * 60)

    try:
        for split in ['train', 'val']:
            try:
                img_path, mask_path = find_sample_pair(DOWNLOAD_DIR, split)
                output_path = os.path.join(DOWNLOAD_DIR, f'sample_{split}.png')
                visualize_sample(img_path, mask_path, output_path=output_path)
            except FileNotFoundError:
                print(f"Не найдено примеров для сплита '{split}'")
    except Exception as e:
        print(f"Ошибка при визуализации: {e}")

    print("\n" + "=" * 60)
    print("Завершение работы")
    print("=" * 60)

    print(f"\nРезультаты:")
    print(f"  - Директория датасета: {DOWNLOAD_DIR}")
    print(f"  - Скачано пакетов: {len(results)}")
    for pkg, info in results.items():
        print(f"    * {pkg}: {info['zip_path']}")

    print(f"\nРекомендации:")
    print(f"  1. Проверьте визуализации в директории: {DOWNLOAD_DIR}")
    print(f"  2. Изучите распределение классов (class_distribution.png)")
    print(f"  3. При необходимости удалите ZIP-архивы для экономии места")
    print(f"  4. CityScapes готов к использованию!")

    print("\nГотово!")

CityScapes Dataset Downloader & Analyzer

[ШАГ 1] Скачивание и распаковка датасета
------------------------------------------------------------
Авторизация успешна

--- Обработка пакета: gtFine ---
Загрузка: Аннотации (маски) высокого качества для train/val/test
URL: https://www.cityscapes-dataset.com/file-handling/?packageID=1
Сохранение в: /content/cityscapes/gtFine_trainvaltest.zip
Прогресс: 100.0%
Загрузка завершена: /content/cityscapes/gtFine_trainvaltest.zip
Распаковка: /content/cityscapes/gtFine_trainvaltest.zip
Целевая директория: /content/cityscapes
Файлов в архиве: 20032
Распаковка завершена. Извлечено элементов: 32

--- Обработка пакета: leftImg8bit ---
Загрузка: Исходные изображения (RGB) для train/val/test
URL: https://www.cityscapes-dataset.com/file-handling/?packageID=3
Сохранение в: /content/cityscapes/leftImg8bit_trainvaltest.zip
Прогресс: 63.6%